#### (번외)불량 유형별 예측

In [ ]:
# y: Defects 그룹 전체 (멀티라벨)
#y_cols = [c for c in df.columns if c[0] == 'Defects']
#y = df[y_cols].astype(int)

# X: Process + Sensor 
#X = df[[c for c in df.columns if c[0] in ('Process', 'Sensor')]].copy()
#X = X.drop(columns=[('Process', 'id')], errors='ignore')
#X = X.drop(columns=[('Process', 'Product_Type')], errors='ignore')

#print("X:", X.shape, "y:", y.shape)
#print("y columns (first 10):", y.columns[:10].tolist())

# 환경설정하기

In [ ]:
# 라이브러리 Import
import numpy as np
import pandas as pd
from datetime import datetime, timedelta
import warnings

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import LabelEncoder, OrdinalEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score, 
                             f1_score, confusion_matrix, classification_report,
                             roc_auc_score, roc_curve)
import xgboost as xgb

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)

# 한글 폰트 설정
import platform
if platform.system() == 'Windows':
    plt.rcParams['font.family'] = 'Malgun Gothic'
elif platform.system() == 'Darwin':
    plt.rcParams['font.family'] = 'AppleGothic'
else:
    plt.rcParams['font.family'] = 'NanumGothic'

plt.rcParams['axes.unicode_minus'] = False
plt.rcParams['figure.figsize'] = (12, 6)

np.random.seed(42)

print("="*60)
print("라이브러리 로드 완료!")
print("="*60)


### 데이터 로드

In [ ]:
df1 = pd.read_csv("../../data/product_type_1.csv", header=[0, 1])
df = df1.copy()
df.info()

In [ ]:
print("="*60)
print("남은 defects 컬럼 확인")
print("="*60)

df[[c for c in df.columns if c[0] == 'defects']].info()

In [ ]:
df['defects'].sum().sort_values(ascending=False)

In [ ]:
defect_groups = {
    "표면": [
        "dent_1",
        "stain_1",
        "exfoliation_1",
        "exfoliation_2"
    ],

    "구조": [
        "short_shot_1",
        "short_shot_2",
        "bubble_1",
        "bubble_2",
        "deformation_1",
        "deformation_2"
    ]
}

In [ ]:

defects = df['defects'].copy()  # columns: Short_Shot_1, Blow_Hole_2, ...

# 각 범주별로 해당하는 defect 컬럼들을 찾아서 "하나라도 1이면 1"로 만들기
y_group = pd.DataFrame(index=df.index)

for group_name, base_names in defect_groups.items():
    # base_names 중 하나로 시작하는 컬럼들 찾기 (예: "Short_Shot" -> "Short_Shot_1", "Short_Shot_2")
    cols = [c for c in defects.columns if any(str(c).startswith(b) for b in base_names)]
    
    if len(cols) == 0:
        # 해당 범주에 매칭되는 컬럼이 없으면 0으로
        y_group[group_name] = 0
    else:
        y_group[group_name] = (defects[cols].fillna(0).astype(int).sum(axis=1) > 0).astype(int)

print(y_group.sum().sort_values(ascending=False))   # 범주별 발생 건수 확인
print(y_group.mean().mul(100))                      # 범주별 발생률(%) 확인

#### 불량 여부 이진 분류

### 타겟 | ('Defect_Flag','Is_Defect')

In [ ]:
print("결측치 상위:\n", df.isnull().sum().sort_values(ascending=False).head(10))

print("\n불량 비율:\n", df[('defect_flag','is_defect')].value_counts(normalize=True))

# Train/Test split

#### XGBoost 사용을 위해 y는 멀티라벨 문제를 제거하는 방향. 
#### 0, 1, 2, 3으로 설정(한 샘플이 표면=1 and/or 구조=1 이 될 수 있음 (멀티라벨) 또는 동시에 0,0 (정상)일 수도 있음) 

In [ ]:
((y_group["표면"]==1) & (y_group["구조"]==1)).sum()

In [ ]:
import re

X = df[['process', 'sensor']].copy()
X = X.drop(columns=[
    ('process', 'id'),
    ('process', 'product_type')
])

# MultiIndex -> 단일 문자열 컬럼명으로 변환
X.columns = [
    f"{lvl0}_{lvl1}" for lvl0, lvl1 in X.columns
]

# 혹시 모를 특수문자 제거
X.columns = [
    re.sub(r"[^A-Za-z0-9_]+", "_", col) for col in X.columns
]
y = pd.Series(np.select(
    [
        (y_group["표면"]==0) & (y_group["구조"]==0),  # 정상
        (y_group["표면"]==1) & (y_group["구조"]==0),  # 표면만
        (y_group["표면"]==0) & (y_group["구조"]==1),  # 구조만
        (y_group["표면"]==1) & (y_group["구조"]==1),  # 복합
    ],
    [0, 1, 2, 3],
    default=0
), name="defect_class")


from sklearn.model_selection import train_test_split


X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print("train:", X_train.shape)
print("test:", X_test.shape)
print("")

print("train 불량률:", y_train.mean())
print("")
print("test 불량률:", y_test.mean())

print("")
print("="*60)
print(f"데이터 분포 유지됨")

print(f"랜덤 split이 잘 됨")

print(f"모델 평가 신뢰도 높음")
print("="*60)


# =================================================
# 여기까지 보시면 됩니다!
# =================================================

In [ ]:
!pip install catboost
!pip install lightgbm
!pip install plotly.express

In [ ]:


from sklearn.ensemble import GradientBoostingClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier
from sklearn.datasets import load_wine
from sklearn.model_selection import StratifiedKFold, cross_val_score, train_test_split
from sklearn.metrics import accuracy_score, classification_report
import plotly.express as px

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

models = {
    "GBM": GradientBoostingClassifier(n_estimators=200, learning_rate=0.1, max_depth=3, random_state=42),
    "XGBoost": XGBClassifier(
        n_estimators=300, learning_rate=0.05, max_depth=4,
        subsample=0.9, colsample_bytree=0.9,
        objective="multi:softprob", num_class=y.nunique(),
        eval_metric="mlogloss", random_state=42
    ),
    "LightGBM": LGBMClassifier(n_estimators=300, learning_rate=0.05, num_leaves=31, random_state=42, verbose=-1),
    "CatBoost": CatBoostClassifier(iterations=300, learning_rate=0.05, depth=6, random_seed=42, verbose=0),
}

results = []
for name, model in models.items():
    cv_scores = cross_val_score(model, X, y, cv=cv, scoring="accuracy", n_jobs=-1)
    model.fit(X_train, y_train)
    test_acc = accuracy_score(y_test, model.predict(X_test))
    results.append({"Model": name, "CV mean": cv_scores.mean(), "CV std": cv_scores.std(), "Test Acc": test_acc})



df_compare = pd.DataFrame(results).sort_values("Test Acc", ascending=False)
display(df_compare)

fig = px.bar(
    df_compare, x="Model", y="Test Acc",
    text="Test Acc",
    title="Boosting 4종 비교 — Test Accuracy (Wine)",
    template="plotly_dark"
)
fig.update_traces(texttemplate="%{text:.3f}", textposition="outside")
fig.update_layout(yaxis=dict(range=[0.85, 1.02]))
fig.show()

In [ ]:
print(y.value_counts())
print(y.value_counts(normalize=True).sort_index())

In [ ]:
from sklearn.dummy import DummyClassifier
from sklearn.model_selection import cross_val_score

dummy = DummyClassifier(strategy="most_frequent")
dummy_scores = cross_val_score(dummy, X, y, cv=cv, scoring="accuracy")

print("Dummy CV mean:", dummy_scores.mean())

In [ ]:
from sklearn.model_selection import cross_validate
from sklearn.metrics import f1_score

cv_result = cross_validate(
    model, X, y, cv=cv,
    scoring=["accuracy", "f1_macro", "f1_weighted"],
    n_jobs=-1
)

In [ ]:
y_pred = model.predict(X_test)

from sklearn.metrics import accuracy_score, f1_score
print("Acc:", accuracy_score(y_test, y_pred))
print("F1-macro:", f1_score(y_test, y_pred, average="macro"))
print("F1-weighted:", f1_score(y_test, y_pred, average="weighted"))

In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt

best_model = XGBClassifier(
    n_estimators=200,
    learning_rate=0.1,
    max_depth=3,
    eval_metric="mlogloss",
    random_state=42
)

best_model.fit(X_train, y_train)
y_pred = best_model.predict(X_test)

cm = confusion_matrix(y_test, y_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm)
disp.plot(cmap="Blues")
plt.show()

In [ ]:
cv_scores = cross_val_score(
    model, X, y, cv=cv, scoring="accuracy",
    n_jobs=-1, error_score="raise"
)

## =========================

## class weight 적용

In [ ]:
from sklearn.utils.class_weight import compute_class_weight
import numpy as np

classes = np.unique(y_train)
weights = compute_class_weight(
    class_weight="balanced",
    classes=classes,
    y=y_train
)

class_weight_dict = dict(zip(classes, weights))

In [ ]:
XGBClassifier(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=4,
    objective="multi:softprob",
    num_class=4,
)

## SMOTE (데이터 균형 맞추기)

In [ ]:
print("결측치 개수 합계:", X_train.isnull().sum().sum()) # train_set.isnull().sum().sum()

print("="*60)
print("결측치 없음!")
print("="*60)

In [ ]:
X_train.dtypes

## X Train data 정보 확인

In [ ]:
print(X_train.shape)
print(X_train.info())


In [ ]:
display(X_train.describe())

In [ ]:
# 박스플롯

cols = X_train.columns

n_cols = 5   # 한 행에 5개
n_rows = (len(cols) // n_cols) + 1

plt.figure(figsize=(20, 4*n_rows))

for i, col in enumerate(cols, 1):
    
    plt.subplot(n_rows, n_cols, i)
    sns.boxplot(y=X_train[col], color='lightgreen')
    plt.title(str(col))
    
plt.tight_layout()
plt.show()

In [ ]:
# Process 관련 컬럼 히스토그램
process_cols = [col for col in X_train.columns if 'process' in str(col)]

plt.figure(figsize=(20, 15))
plt.suptitle("공정(Process) 데이터 분포 및 이상치 확인", fontsize=20)

for i, col in enumerate(process_cols, 1):
    plt.subplot(4, 4, i)
    sns.histplot(X_train[col], kde=True, color='skyblue')
    plt.title(f'Dist: {col[1]}') # MultiIndex의 하위 컬럼명 사용
    plt.tight_layout(rect=[0, 0.03, 1, 0.95])

plt.show()

# Sensor 관련 컬럼 히스토그램
sensor_cols = [col for col in X_train.columns if 'sensor' in str(col)]

plt.figure(figsize=(20, 15))
plt.suptitle("센서(Sensor) 데이터 분포 및 이상치 확인", fontsize=20)

for i, col in enumerate(sensor_cols, 1):
    plt.subplot(4, 4, i)
    sns.histplot(X_train[col], kde=True, color='salmon')
    plt.title(f'Dist: {col[1]}')
    plt.tight_layout(rect=[0, 0.03, 1, 0.95])

plt.show()

#### feature 간 상관관계 - multicollinearity 확인

In [ ]:
corr = X_train.corr()

plt.figure(figsize=(12,10))

sns.heatmap(
    corr,
    cmap="YlGnBu",
    annot=True,      
    fmt=".2f"        
)

plt.show()

## y 타겟(불량) 분포 확인 

In [ ]:
y_train.value_counts(normalize=True)

In [ ]:
# 정상 불량 분포 확인하기


print("="*60)
print("Class imbalance 확인")
print("="*60)

sns.countplot(x=y_train)
plt.title("Defect Distribution")
plt.show()



### 간단한 RandomForest로 중요 변수 확인해보기

In [ ]:
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(n_estimators=200, random_state=42)

rf.fit(X_train, y_train)

import pandas as pd

importance = pd.Series(rf.feature_importances_, index=X_train.columns)
importance.sort_values(ascending=False).head(10)